## Fuel Rate - Raw to Bronze Processing

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

schema = StructType([
    StructField("Fuel Station ID", StringType(), False),
    StructField("Fuel Rate ID", StringType(), False),
    StructField("Fuel Type", StringType(), False),
    StructField("Fuel Cost", DoubleType(), False),
    StructField("Start DateTime", TimestampType(), False),
    StructField("End DateTime", TimestampType(), False)
])

In [0]:
TARGET_TABLE = "fuel_project_dev.bronze.fuel_rates"

In [0]:
'''
Fuel Rate data:
Drives pricing
Used in revenue calculations
Small but business critical
Silent schema drift here = financial corruption.
Failing fast is good.

What If Additional Columns Later?

- Update schema definition
- Version the Bronze table
- Backfill if required

This is controlled change, not accidental drift.
'''

In [0]:
df = (spark.read.format("csv")
      .schema(schema)
      .option("header", True)
      .option("mode","FAILFAST")
      .load("/Volumes/fuel_project_dev/raw/fuel_rates/*.csv"))

In [0]:
df = df.withColumn("_bronze_ingestion_timestamp", current_timestamp()) \
  .withColumn("_bronze_source_file_path", col("_metadata.file_path"))

df = df.withColumnsRenamed({"Fuel Station ID": "fuel_station_id",
                       "Fuel Rate ID": "fuel_rate_id",
                       "Fuel Type": "fuel_type",
                       "Fuel Cost": "fuel_cost",
                       "Start DateTime": "start_datetime",
                       "End DateTime": "end_datetime"})

In [0]:
df.write.mode("append").saveAsTable(TARGET_TABLE)